## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

print("Bibliotheques importees.")

Bibliotheques importees.


## Parametres

In [2]:
INPUT_FILE   = "../../data/movies/movie_data.csv"
TARGET_COL   = "Views"
RANDOM_STATE = 42
TEST_SIZE    = 0.2

OUTPUT_DIR = "../../outputs/movies"

## Chargement et nettoyage des données

In [3]:
print("[SVM] Chargement du fichier source...")
data = pd.read_csv(INPUT_FILE, sep=";")

nb_na = data.isna().sum().sum()
print(f"[SVM] Valeurs manquantes detectees : {nb_na}")
data = data.dropna().reset_index(drop=True)

print(f"[SVM] Dimensions apres verification : {data.shape}")
data.head()

[SVM] Chargement du fichier source...
[SVM] Valeurs manquantes detectees : 0
[SVM] Dimensions apres verification : (187, 8)


,Gross,Budget,Screens,Views,Likes,Dislikes,Comments,Aggregate Followers
0,9130,4000000.0,45.0,3280543,4632,425,636,1120000.0
1,192000000,50000000.0,3306.0,583289,3465,61,186,12350000.0
2,30700000,28000000.0,2872.0,304861,328,34,47,483000.0
3,106000000,110000000.0,3470.0,452917,2429,132,590,568000.0
4,17300000,3500000.0,2310.0,3145573,12163,610,1082,1923800.0


## Séparation train/test (80/20)

In [4]:
X = data.drop(columns=[TARGET_COL])
y = data[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"[SVM] Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

X_train.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False, sep=";")
X_test.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False, sep=";")
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False, sep=";")
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False, sep=";")
print("[SVM] Splits sauvegardes dans outputs/movies/")

[SVM] Train : 149 lignes | Test : 38 lignes
[SVM] Splits sauvegardes dans outputs/movies/


## Entraînement du SVM

In [5]:
print("[SVM] Entrainement du modele SVR (noyau RBF, hyperparametres par defaut)...")

base_pipeline = make_pipeline(StandardScaler(), SVR(kernel="rbf"))
model = TransformedTargetRegressor(regressor=base_pipeline, transformer=StandardScaler())
model.fit(X_train, y_train)

print("[SVM] Entrainement termine.")

[SVM] Entrainement du modele SVR (noyau RBF, hyperparametres par defaut)...
[SVM] Entrainement termine.


## Évaluation

In [7]:
print("[SVM] Evaluation sur l'ensemble de test (holdout)...")

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"[SVM] RMSE : {rmse:.2f}")
print(f"[SVM] MAE  : {mae:.2f}")
print(f"[SVM] R2   : {r2:.4f}")
print(f"[SVM] MAPE : {mape:.2f}%")

pd.DataFrame([{"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}]).to_csv(
    f"{OUTPUT_DIR}/metriques_svm.csv", index=False, sep=";"
)

[SVM] Evaluation sur l'ensemble de test (holdout)...
[SVM] RMSE : 1573803.30
[SVM] MAE  : 1216567.47
[SVM] R2   : 0.8028
[SVM] MAPE : 392.07%


## Sauvegarde du modèle

In [8]:
joblib.dump(model, f"{OUTPUT_DIR}/svm_model.pkl")
print(f"[SVM] Modele sauvegarde dans {OUTPUT_DIR}/svm_model.pkl")
print("[SVM] Ce notebook n'a besoin d'etre execute qu'UNE SEULE FOIS.")

[SVM] Modele sauvegarde dans ../../outputs/movies/svm_model.pkl
[SVM] Ce notebook n'a besoin d'etre execute qu'UNE SEULE FOIS.
